----
## <font color='CornflowerBlue'>Practical 5: Self-supervised training</font> 
##### Alok Bharadwaj and Arjen Jakobi
---

## 5.8: Self-supervised training with language models

In all previous weeks, training required a labelled dataset: an input paired with a known target. **Self-supervised learning** removes this requirement. Instead of manually labelled targets, the model generates its own supervision signal directly from the raw data.

### Why self-supervised learning?

The key motivation is **scale**. Labelled biological datasets are expensive to produce. A single experimentally solved protein structure can take years of work, but *unlabelled* sequence data is available in vast quantities. Self-supervised training lets us exploit this abundance.

To appreciate the scale involved, here are approximate token counts for common training corpora:

| Dataset | Approximate size |
|---------|------------------|
| English Wikipedia | ~4 billion tokens |
| Common Crawl (filtered, e.g. C4) | ~150 billion tokens |
| UniRef50 protein sequences | ~17 billion amino acid tokens |
| GPT-3 full training set | ~300 billion tokens |

For comparison, a person who reads one book per week for 50 years would encounter roughly 150 million words. This is about 1,000× less than a typical large language model sees during training. This is both remarkable and thought-provoking: what does it mean to 'understand' language if you have read vastly more text than any human ever could?

### Two self-supervised training objectives

**Causal Language Modelling (CLM):** Predict the probability distribution of the *next* token given all preceding tokens. The model sees only left-context (enforced by a causal attention mask). This is the pre-training objective for GPT-style generative models.

**Masked Language Modelling (MLM):** Randomly mask a fraction of tokens and predict their identity from *both* left and right context. This is the pre-training objective for BERT-style encoder models and for most protein language models (ESM, ProtTrans).

This notebook covers both objectives. We start with some standard metrics for assessing model quality before implementing training.


### 5.8.1: Loss function and perplexity

The standard loss function for language model training is **cross-entropy loss**. At each predicted position, the model outputs a probability distribution over the entire vocabulary. The loss penalises the model for assigning low probability to the correct token:

\begin{equation*}
    \mathcal{L} = -\frac{1}{N} \sum_{i=1}^{N} \log P(w_i \mid \text{context}_i)
\end{equation*}

where $N$ is the number of predicted tokens and $\text{context}_i$ is the input visible to the model at position $i$ (all preceding tokens for CLM; all unmasked tokens for MLM).

#### Perplexity

**Perplexity** (PP) is a human-interpretable metric derived directly from the cross-entropy loss:

\begin{equation*}
    \text{PP} = \exp\!\left(-\frac{1}{N} \sum_{i=1}^{N} \log P(w_i \mid \text{context}_i)\right) = \exp(\mathcal{L})
\end{equation*}

Intuitively, a perplexity of $k$ means the model is about as uncertain as if it were choosing uniformly among $k$ equally likely options at each step:

- **PP = 1**: perfect prediction — the model is never surprised.
- **PP = vocab_size**: random guessing — the model assigns equal probability to every token.
- **PP < vocab_size**: the model has learned something useful from the data.

For our Shakespeare character-level model (vocab size 65), a random model would yield PP ≈ 65. A well-trained model should achieve PP well below 10.

Run the code cell below to visualise the relationship between cross-entropy loss and perplexity.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

vocab_size = 65  # Shakespeare character-level vocabulary

# Perplexity of a random model equals the vocabulary size
random_loss = np.log(vocab_size)
print(f"Random-model cross-entropy loss: {random_loss:.2f}")
print(f"Random-model perplexity:          {np.exp(random_loss):.1f}  (= vocab size {vocab_size})")

# Plot perplexity as a function of loss
losses = np.linspace(0, 5, 200)
perplexities = np.exp(losses)

plt.figure(figsize=(7, 4))
plt.plot(losses, perplexities, lw=2)
plt.axvline(random_loss, color='r', linestyle='--',
            label=f'Random baseline (PP = {vocab_size})')
plt.axhline(vocab_size, color='r', linestyle=':', alpha=0.5)
plt.xlabel('Cross-entropy loss')
plt.ylabel('Perplexity')
plt.title('Perplexity vs. cross-entropy loss')
plt.legend()
plt.tight_layout()
plt.show()


### 5.3.2: Training a Language Model for text generation. 

In the src/lm_utils.py file we have defined the Language Model class as it was written in Module 2. We have left out one function which we need to train the language model for text generation. 

#### Text generation

Generating text from a language model is done by sequentially predicting embedding vector for the next token given an input sequence. We do this by employing a causal mask in the attention head. 

In [ ]:
shakespeare_path = "tinyshakespeare.txt"
with open(shakespeare_path, "r") as f:
    text = f.read()
print(f"Length of text: {len(text)} characters")


Now, get all the unique characters to get a vocabulary


In [ ]:
unique_chars = sorted(set(text))
print(f"Unique characters: {''.join(unique_chars)}")
vocab_size = len(unique_chars)  
print(f"Vocabulary size: {vocab_size}")

We tokenise the chacters by the index in the vocabulary 

In [ ]:
encode = {ch: i for i, ch in enumerate(unique_chars)}
decode = {i: ch for i, ch in enumerate(unique_chars)}

sample_text = "Hello world!"
encoded = [encode[ch] for ch in sample_text]
print(f"Encoded: {encoded}")
decoded = ''.join(decode[i] for i in encoded)
print(f"Decoded: {decoded}")


In [ ]:
import torch

data = torch.tensor([encode[ch] for ch in text], dtype=torch.long)
percent_training = 0.9
n = int(percent_training * len(data))
train_data = data[:n]
val_data = data[n:]
print(f"Training data length: {len(train_data)}")
print(f"Validation data length: {len(val_data)}")


In [ ]:
# Create batches of data for training using sequential sampling
def get_batch(data, batch_size, block_size):
    # Randomly select starting indices for the batch
    start_indices = torch.randint(0, len(data) - block_size, (batch_size,))
    
    # Create input and target batches
    x = torch.stack([data[i:i+block_size] for i in start_indices])
    y = torch.stack([data[i+1:i+block_size+1] for i in start_indices])
    
    return x, y

In [ ]:
x_batch, y_batch = get_batch(train_data, batch_size=4, block_size=8)
print("Input batch (x):")
print(x_batch)
print("Target batch (y):")
print(y_batch)


In [ ]:
from src.lm_utils import LanguageModel

n_layers = 4 
n_heads = 4
d_input = 64
context_length = 32
batch_size = 16
d_model = 64

tinyLM = LanguageModel(
    n_layers=n_layers,
    n_head=n_heads,
    d_input=d_input,
    d_model=d_model,
    context_length=context_length,
    vocab_size=vocab_size,
    d_ff=d_model*4
)



In [ ]:
import torch 
import torch.nn as nn

sample_idx = torch.tensor([[1, 3, 4, 5]])
logits, attns = tinyLM(sample_idx, causal=True, return_attention_weights=True)
print(logits.shape)  # Should be (batch_size, seq_length, vocab_size)

In [ ]:
generated = tinyLM.generate(sample_idx, max_new_tokens=5)
print("Generated indices:", generated)
generated_text = ''.join(decode[idx.item()] for idx in generated[0])
print("Generated text:", generated_text)

### 5.3.2 Setting up training 

In [ ]:
learning_rate = 1e-3
max_epochs = 1000
optimizer = torch.optim.Adam(tinyLM.parameters(), lr=learning_rate)
loss = nn.CrossEntropyLoss()


In [ ]:
def evaluate(model, data, block_size):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for i in range(0, len(data) - block_size, block_size):
            x = data[i:i+block_size].unsqueeze(0)  # Add batch dimension
            y = data[i+1:i+block_size+1].unsqueeze(0)
            logits = model(x, causal=True)
            total_loss += loss(logits.view(-1, vocab_size), y.view(-1)).item()
    return total_loss / (len(data) // block_size)

def generate_text(model, start_text="To be or", max_new_tokens=20):
    model.eval()
    input_indices = torch.tensor([[encode[ch] for ch in start_text]], dtype=torch.long)
    generated_indices = model.generate(input_indices, max_new_tokens=max_new_tokens)
    generated_text = ''.join(decode[idx.item()] for idx in generated_indices[0])
    print(generated_text)
    


Now, run the next code cell with the default parameters. After 100 epochs, the loop outputs training and validation loss. It also generates new text from a starting prompt. Notice how the text generation changes with more training. 

In [ ]:
for epoch in range(max_epochs):
    tinyLM.train()
    x_batch, y_batch = get_batch(train_data, batch_size, context_length)
    optimizer.zero_grad()
    logits = tinyLM(x_batch, causal=False) 
    train_loss = loss(logits.view(-1, vocab_size), y_batch.view(-1))
    train_loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        val_loss = evaluate(tinyLM, val_data, context_length)
        print(f"Epoch {epoch}, Train Loss: {train_loss.item():.4f}, Val Loss: {val_loss:.4f}")
        generate_text(tinyLM, start_text="Hello", max_new_tokens=20)

Notice the training and validation losses. If you ran the previous code cell with default parameters, then you should see a rapid *decrease* in training loss while the validation loss remains constant. This is a classic sign of overfitting. You may also notice the generated text do not show any improvement during learning. 

Think for a while. What small change could we make to the code to improve the output? If you want to re-run the training, you need to re-initialise the model so that it is not biased by current training. Run the next code cell to get longer generation. 

In [ ]:
# After training, we can generate some text to see how well the model has learned
prompt = "To be, or not"
prompt_idx = torch.tensor([[encode[ch] for ch in prompt]], dtype=torch.long)
generated = tinyLM.generate(prompt_idx, max_new_tokens=100, top_k=10)
generated_text = ''.join(decode[idx.item()] for idx in generated[0])
print("Generated text:")
print("-" * 40)
print(generated_text)

If you managed to correct the error, you might see that the generated sequence starts to look a lot more familiar. Here is the sort of output expected (for the same starting prompt):

---

_To be, or nothet blet, bus for hent her;_

_And host takir at tutis them hene_

_And, dowe houn deses, do for sh thes c_

---

The words starts to get shape. The repetetive feature would also disappear. However, the words might still not make much sense. That is because we are training at a character-level encoding. Each character is not very expressive. To see what happens in a more expressive model, check out the notebook [tinyLanguageModel.ipynb](./4_tinyLanguageModel.ipynb). Here we have implemented Byte-Pair Encoding using the same tokeniser used in GPT-2. The vocabulary size increases and each token can more meaningfully inform the context due to increased expression. 

Train the new model using a GPU. You do not have to do it now. You can also use any text you want (as long as the GPT2 tokeniser can handle it). If we have enough trained models by next week, we can chain their generated outputs together and watch their interaction. Basically, we can watch robots talk to each other! 

### 5.3.4: Masked language modelling on protein sequences

Causal language modelling (Section 5.3.2) predicts tokens left-to-right, which is natural for text generation. For proteins, however, we want the model to use **bidirectional context**. A residue's identity depends on what flanks it on both sides of the sequence, not only what precedes it.

**Masked Language Modelling (MLM)** achieves this. During training:
1. A random subset of residues (typically ~15%) is replaced with a special `[MASK]` token.
2. The model predicts the original identity of each masked residue from the remaining unmasked context.
3. Because the model sees both left and right context simultaneously, no causal mask is applied.

This is the training strategy used by **BERT** for language and by **ESM** for proteins. By forcing the model to reconstruct masked residues from context, it learns the statistical grammar of protein sequences, i.e. which amino acids tend to co-occur and in what structural environments, purely from sequence data, without any access to 3D structure labels.

The code below loads a dataset of protein sequences and sets up the MLM training pipeline.


In [ ]:
import torch
import torch.nn as nn

sequences_path = "protein_dataset.txt"
with open(sequences_path, "r") as f:
    sequences = f.read()


print(f"Length of sequences: {len(sequences)} characters")


In [ ]:
# Create a vocabulary of unique characters in the sequences
unique_chars = sorted(set(sequences))
print(f"Unique characters: {''.join(unique_chars)}")
vocab_size = len(unique_chars)
print(f"Vocabulary size: {vocab_size}")
# Create a mask token 
mask_token = "#"
# Add the mask token to the vocabulary if it's not already present
if mask_token not in unique_chars:
    unique_chars.append(mask_token)
    vocab_size += 1
    print(f"Added mask token '{mask_token}' to vocabulary. New vocab size: {vocab_size}")

In [ ]:
encode = {ch: i for i, ch in enumerate(unique_chars)}
decode = {i: ch for i, ch in enumerate(unique_chars)}

# Encode a sample sequence
sample_sequence = "MKTIIALSYIFCLVFADYKDDDDK"
encoded = [encode[ch] for ch in sample_sequence]
print(f"Encoded: {encoded}")
# Decode the encoded sequence
decoded = ''.join(decode[i] for i in encoded)
print(f"Decoded: {decoded}")

In [ ]:
plm_data = torch.tensor([encode[ch] for ch in sequences], dtype=torch.long)
percent_training = 0.9
n = int(percent_training * len(plm_data))
train_data = plm_data[:n]
val_data = plm_data[n:]
print(f"Training data length: {len(train_data)}")
print(f"Validation data length: {len(val_data)}")


In [ ]:
from src.lm_utils import LanguageModel

n_layers_plm = 12
n_heads_plm = 8
d_input_plm = 128
d_model_plm = 128
context_length_plm = 32
proteinLM = LanguageModel(
    n_layers=n_layers_plm,
    n_head=n_heads_plm,
    d_input=d_input_plm,
    d_model=d_model_plm,
    context_length=context_length_plm,
    vocab_size=vocab_size,
    d_ff=d_model_plm*4
)


In [ ]:
def evaluate_mlm_accuracy(model, data, batch_size, block_size, n_batches=50, mask_ratio=0.15):
    """Fraction of masked tokens the model predicts correctly (validation)."""
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for _ in range(n_batches):
            x_batch, y_batch = get_batch_mlm(data, batch_size, block_size, mask_ratio=mask_ratio)
            logits = model(x_batch, causal=False)          # (B, T, vocab)
            preds = logits.argmax(dim=-1)                  # (B, T) most-likely token

            masked = y_batch != -100                       # only the masked positions count
            correct += (preds[masked] == y_batch[masked]).sum().item()
            total   += masked.sum().item()
    model.train()
    return correct / max(total, 1) * 100.0

# Create batches of data for training using sequential sampling
def get_batch_mlm(data, batch_size, block_size, mask_ratio=0.15):
    # Randomly select starting indices for the batch
    start_indices = torch.randint(0, len(data) - block_size, (batch_size,))
    
    # Create input and target batches
    x = torch.stack([data[i:i+block_size] for i in start_indices])
    
    # special-token IDs that must never be masked or predicted
    special_ids = torch.tensor([encode["^"], encode["$"], encode["\n"]])
    is_special = torch.isin(x, special_ids)

    # choose positions to mask: random, but never a special token
    mask = (torch.rand(x.shape) < mask_ratio) & (~is_special)

    x = x.masked_fill(mask, encode['#'])  # Replace masked positions with 'X' token
    y = torch.stack([data[i:i+block_size] for i in start_indices])
    y = y.masked_fill(~mask, -100)  # Only compute loss on masked positions

    
    return x, y

In [ ]:
learning_rate_plm = 1e-4
max_epochs_plm = 1000
optimizer_plm = torch.optim.Adam(proteinLM.parameters(), lr=learning_rate_plm)
loss_plm = nn.CrossEntropyLoss(ignore_index=-100)
batch_size = 16

In [ ]:
for epoch in range(max_epochs_plm):
    proteinLM.train()
    x_batch, y_batch = get_batch_mlm(train_data, batch_size, context_length_plm, 
                                     mask_ratio=0.5)
    optimizer_plm.zero_grad()
    logits = proteinLM(x_batch, causal=False)
    train_loss = loss_plm(logits.view(-1, vocab_size), y_batch.view(-1))
    train_loss.backward()
    optimizer_plm.step()
    
    if epoch % 100 == 0:
        acc = evaluate_mlm_accuracy(proteinLM, val_data, batch_size, context_length_plm, mask_ratio=0.5)
        print(f"Epoch {epoch}, Train Loss: {train_loss.item():.4f}, Val Acc: {acc:.1f}%")
        

### Assignment 3 (bonus): Training and evaluating your protein language model

In the following notebook you will find scaffolding code to train a small protein language model with BPE tokenisation.

1. **Run the baseline training** with the default hyperparameters. Record the training and validation loss curves and note the final perplexity.

2. **Experiment with at least two hyperparameter changes.** Good candidates include: number of transformer layers, embedding dimension ($d_{\text{input}}$), learning rate, masking ratio, or context length. For each run, note what you changed and whether performance improved or worsened.

3. **Save your trained model** to the shared folder */projects/nb4170/protein_language_models/*:

```python
torch.save(proteinLM.state_dict(), "/projects/nb4170/protein_language_models/your_name_proteinLM.pt")
```

4. **Reflection** (answer in a markdown cell):
   - What is the final validation accuracy on masked token prediction? How does this compare to random guessing?
   - What are the main factors limiting performance of this small model?
   - ESM-2 (650 M parameters) achieves near-perfect reconstruction of masked residues. What would need to change — in model size, training data, or training duration — to approach that level of performance?
